# Issue #250 source census — PI takeover decision notebook

This is an **assessment-only** view. It profiles existing files and never crawls, backfills, imputes, trains, builds tickets, or computes ROI. The key distinction is between file presence, a scoped historical relative-time proxy, and canonical causal model eligibility.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
from source_census.census import PACKAGE_NAME, PROXY_POLICY_ID

package = root / 'results' / PACKAGE_NAME
assert package.is_dir(), package
print(f'Package: {package.relative_to(root)}')
print(f'Historical proxy policy: {PROXY_POLICY_ID}')
print('Trust note: independent verification still requires an external anchor.')

Package: results/bj250_source_census_v3
Historical proxy policy: ISSUE_250_PI_USER_V2_RELATIVE_PROXY
Trust note: independent verification still requires an external anchor.


In [2]:
def table(name: str) -> pd.DataFrame:
    return pd.read_csv(package / 'tables' / f'{name}.csv')

roles = table('source_role_matrix')
print('SOURCE ROLES')
print(roles[['source_id', 'lane', 'role', 'decision']].to_string(index=False))

SOURCE ROLES
           source_id                           lane             role                               decision
         beidan_xlsm historical_relative_time_proxy HISTORICAL_PROXY       ASSESSMENT_ONLY_NO_POLICY_FREEZE
     beidan_features      identity_schedule_support         RAW_ONLY       ASSESSMENT_ONLY_NO_POLICY_FREEZE
      beidan_history         settlement_label_truth  SETTLEMENT_ONLY       ASSESSMENT_ONLY_NO_POLICY_FREEZE
         beidan_live       strict_forward_candidate         RAW_ONLY SEAL_SOURCE_AND_CAPTURE_CLOCK_CRITERIA
    bjdc_500_capture       strict_forward_candidate         RAW_ONLY SEAL_SOURCE_AND_CAPTURE_CLOCK_CRITERIA
beidan_modeling_plus                   derived_copy          EXCLUDE       ASSESSMENT_ONLY_NO_POLICY_FREEZE
     beidan_nowscore                   derived_copy          EXCLUDE       ASSESSMENT_ONLY_NO_POLICY_FREEZE


In [3]:
surface = table('proxy_surface_profile')
cols = ['market', 'snapshot_horizon', 'vector_semantics', 'admissible_use',
        'role_gate_eligible', 'vector_row_count', 'physical_cell_count',
        'non_null_physical_cell_count', 'native_identity_valid_match_count']
print('HISTORICAL RELATIVE-TIME SURFACES')
print(surface.loc[surface.source_id.eq('beidan_xlsm'), cols].to_string(index=False))
print()
print('Interpretation: only Euro 1X2 is a three-outcome role-gate surface. Asian handicap and over/under are two prices plus a line, not p34 outcome triplets.')

HISTORICAL RELATIVE-TIME SURFACES
        market snapshot_horizon                            vector_semantics               admissible_use  role_gate_eligible  vector_row_count  physical_cell_count  non_null_physical_cell_count  native_identity_valid_match_count
asian_handicap              10h     TWO_PRICE_PLUS_LINE_NOT_OUTCOME_TRIPLET     AUXILIARY_RELATIVE_PROXY               False           3514842             10544526                       7150563                             188048
      euro_1x2              10h                      THREE_OUTCOME_EURO_1X2   HISTORICAL_PROXY_CANDIDATE                True           2929035              8787105                       7469674                             189611
     kelly_1x2              10h                 DERIVED_THREE_OUTCOME_KELLY HISTORICAL_PROXY_DESCRIPTIVE               False           2929035              8787105                       5315526                             133033
    over_under              10h TWO_PRICE_PLUS_LIN

In [4]:
period = table('proxy_coverage_by_period')
focus = period.loc[period.year_or_range.astype(str).eq('2024')]
cols = ['market', 'snapshot_horizon', 'match_denominator', 'issue_denominator',
        'vector_row_count', 'physical_cell_count', 'non_null_physical_cell_count',
        'complete_vector_row_count', 'missing_match_count']
print('2024 PERIOD × SURFACE COVERAGE (real denominators)')
print(focus[cols].to_string(index=False))
print()
print('The rows above deliberately separate matches, vector rows, and physical cells.')

2024 PERIOD × SURFACE COVERAGE (real denominators)
        market snapshot_horizon  match_denominator  issue_denominator  vector_row_count  physical_cell_count  non_null_physical_cell_count  complete_vector_row_count  missing_match_count
asian_handicap              10h              16192                 67            291456               874368                        680064                     226450                    0
      euro_1x2              10h              16192                 67            242880               728640                        650868                     216676                    0
     kelly_1x2              10h              16192                 67            242880               728640                        650868                     216676                    0
    over_under              10h              16192                 67            291456               874368                        837001                     278731                    0
asian_handicap

In [5]:
overrides = table('hist40_scope_overrides')
print('INHERITED R8 FIELD × MARKET PROHIBITIONS')
print(overrides[['table', 'column', 'market', 'admissible_use', 'provenance_grade']].to_string(index=False))

intersections = table('intersection_levels')
print()
print('FOUR INTERSECTION LEVELS')
print(intersections[['level', 'lower_bound', 'upper_bound', 'authoritative', 'decision']].to_string(index=False))

INHERITED R8 FIELD × MARKET PROHIBITIONS
      table column        market admissible_use                   provenance_grade
exotic_odds   odds correct_score     PROHIBITED         UNVERIFIED_MARKET_AND_TIME
exotic_odds   odds   total_goals     PROHIBITED HISTORICAL_P40_SCHEMA_INCOMPATIBLE

FOUR INTERSECTION LEVELS
                        level                            lower_bound                            upper_bound  authoritative                                       decision
                  RAW_PRESENT initial/10h/1h field families profiled initial/10h/1h field families profiled           True                          HISTORICAL_PROXY_ONLY
      NATIVE_KEY_INTERSECTION                                   1748                                   1748           True                              EXACT_NATIVE_ONLY
       CANDIDATE_MAPPED_RANGE                                 188197                                 188199          False NON_AUTHORITATIVE_PROPOSED_MAPPING_PENDING_251
CANO

In [6]:
availability = table('source_feature_availability')
print('SOURCE × FEATURE AVAILABILITY')
print(availability[['source_id', 'field_family', 'r8_generic_admissible_field_count',
                    'issue250_scoped_surface_evidence_count', 'present', 'role']].to_string(index=False))

m = table('m_per_issue_distribution')
summary = m.groupby('source_id').m_matches.agg(['count', 'min', 'median', 'max']).reset_index()
print()
print('M PER ISSUE')
print(summary.to_string(index=False))

SOURCE × FEATURE AVAILABILITY
           source_id      field_family  r8_generic_admissible_field_count  issue250_scoped_surface_evidence_count  present             role
         beidan_xlsm identity_schedule                                  0                                       1     True HISTORICAL_PROXY
         beidan_xlsm         timestamp                                  0                                       3     True HISTORICAL_PROXY
         beidan_xlsm    market_feature                                  1                                      12     True HISTORICAL_PROXY
         beidan_xlsm  label_settlement                                  0                                       0    False HISTORICAL_PROXY
     beidan_features identity_schedule                                  0                                       0    False         RAW_ONLY
     beidan_features         timestamp                                  0                                       0    False        

In [7]:
ledger = table('unresolved_question_ledger')
print('DECISION SUMMARY')
print('1. beidan_xlsm is usable only as an accepted retrospective relative-time proxy surface.')
print('2. File availability is not canonical causal eligibility; canonical overlap stays deferred to #251/#252.')
print('3. Both live lanes remain raw/unfrozen until clock and sealing policy is verified.')
print('4. No crawling or data recovery was performed in #250.')
print()
print('UNRESOLVED QUESTIONS')
print(ledger.to_string(index=False))

DECISION SUMMARY
1. beidan_xlsm is usable only as an accepted retrospective relative-time proxy surface.
2. File availability is not canonical causal eligibility; canonical overlap stays deferred to #251/#252.
3. Both live lanes remain raw/unfrozen until clock and sealing policy is verified.
4. No crawling or data recovery was performed in #250.

UNRESOLVED QUESTIONS
        owner_issue                                                  question question_id               status
               #251                 Which proposed mappings become canonical?          U1             DEFERRED
               #252 What sealing/source-clock rules promote forward evidence?          U2             DEFERRED
#235/#236/#237/#248                     Which measured gaps justify recovery?          U3 PI_DECISION_REQUIRED
